# Energy Demand Forecasting using LSTM

This notebook builds an **LSTM** model for hourly energy demand forecasting.

## Goals
- Clean the dataset in a consistent way
- Create time-based features and lag features
- Use a **chronological split** to avoid data leakage
- Scale using **training data only**
- Evaluate on the **original demand values**
- Compare fairly with ARIMA / SARIMA

> Expected file: `energy_dataset.csv`


In [ ]:
# Uncomment this only if the libraries are not installed yet
# !pip install pandas numpy matplotlib scikit-learn torch


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error


## 1. Load and clean the dataset

This section uses the same target and similar cleaning logic as the ARIMA / SARIMA notebook.


In [ ]:
# Load the dataset
df = pd.read_csv("energy_dataset.csv")

# Parse and clean the time column
df["time"] = pd.to_datetime(df["time"], utc=True, errors="coerce")
df = df.dropna(subset=["time"]).copy()
df = df.sort_values("time")
df = df.set_index("time")

# Remove duplicate timestamps
df = df[~df.index.duplicated(keep="first")]

# Resample to hourly frequency for consistent time steps
df = df.resample("H").mean()

# Define the target
target_col = "total load actual"

# Fill missing values in all numeric columns
df = df.interpolate(method="time").ffill().bfill()

# Remove obvious target outliers with IQR, then refill
q1 = df[target_col].quantile(0.25)
q3 = df[target_col].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

df.loc[(df[target_col] < lower) | (df[target_col] > upper), target_col] = np.nan
df[target_col] = df[target_col].interpolate(method="time").ffill().bfill()

print("Cleaned shape:", df.shape)
df.head()


## 2. Feature engineering

We build:
- time features
- lag features
- selected generation features

These features help the LSTM learn demand patterns better.


In [ ]:
# Time-based features
df["hour"] = df.index.hour
df["dayofweek"] = df.index.dayofweek
df["month"] = df.index.month

# Lag features from the target
df["lag_1"] = df[target_col].shift(1)
df["lag_24"] = df[target_col].shift(24)

# Choose input features
candidate_features = [
    "generation solar",
    "generation wind onshore",
    "generation wind offshore",
    "generation nuclear",
    "generation fossil gas",
    "hour",
    "dayofweek",
    "month",
    "lag_1",
    "lag_24"
]

# Keep only features that really exist in the file
features = [col for col in candidate_features if col in df.columns]

# Build the working dataframe
model_df = df[features + [target_col]].copy()

# Drop rows created by lagging
model_df = model_df.dropna().copy()

print("Features used:")
print(features)
print("\nModel dataframe shape:", model_df.shape)
model_df.head()


## 3. Chronological train-test split

We use the **last 20%** as test data.
This matches the time-series logic used in the ARIMA / SARIMA notebook.


In [ ]:
split_idx = int(len(model_df) * 0.80)

train_df = model_df.iloc[:split_idx].copy()
test_df = model_df.iloc[split_idx:].copy()

print("Train size:", len(train_df))
print("Test size:", len(test_df))
print("Train range:", train_df.index.min(), "to", train_df.index.max())
print("Test range:", test_df.index.min(), "to", test_df.index.max())


## 4. Scale using training data only

This is very important.
We **fit** the scaler only on training data to prevent data leakage.


In [ ]:
feature_scaler = MinMaxScaler()
target_scaler = MinMaxScaler()

# Fit on training data only
X_train_scaled = feature_scaler.fit_transform(train_df[features])
X_test_scaled = feature_scaler.transform(test_df[features])

y_train_scaled = target_scaler.fit_transform(train_df[[target_col]])
y_test_scaled = target_scaler.transform(test_df[[target_col]])

print("Scaled training feature shape:", X_train_scaled.shape)
print("Scaled test feature shape:", X_test_scaled.shape)


## 5. Build sequences

Each sample uses the previous `seq_length` time steps to predict the next demand value.


In [ ]:
def create_sequences(X, y, seq_length=48):
    X_seq, y_seq = [], []
    for i in range(len(X) - seq_length):
        X_seq.append(X[i:i + seq_length])
        y_seq.append(y[i + seq_length])
    return np.array(X_seq), np.array(y_seq)

seq_length = 48

X_train_seq, y_train_seq = create_sequences(X_train_scaled, y_train_scaled, seq_length=seq_length)
X_test_seq, y_test_seq = create_sequences(X_test_scaled, y_test_scaled, seq_length=seq_length)

print("X_train_seq:", X_train_seq.shape)
print("y_train_seq:", y_train_seq.shape)
print("X_test_seq :", X_test_seq.shape)
print("y_test_seq :", y_test_seq.shape)


## 6. Convert data to PyTorch tensors


In [ ]:
X_train_t = torch.tensor(X_train_seq, dtype=torch.float32)
y_train_t = torch.tensor(y_train_seq, dtype=torch.float32)

X_test_t = torch.tensor(X_test_seq, dtype=torch.float32)
y_test_t = torch.tensor(y_test_seq, dtype=torch.float32)

X_train_t.shape, y_train_t.shape


## 7. Define the LSTM model

This version fixes the typo in `dropout` and uses:
- 2 LSTM layers
- dropout for regularization
- a final linear layer for prediction


In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]   # Use the last time step
        out = self.fc(out)
        return out

model = LSTMModel(input_size=len(features))
model


## 8. Train the model


In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 20
train_losses = []

for epoch in range(epochs):
    model.train()

    outputs = model(X_train_t)
    loss = criterion(outputs, y_train_t)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    train_losses.append(loss.item())

    print(f"Epoch {epoch + 1}/{epochs} - Loss: {loss.item():.6f}")


## 9. Plot the training loss


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(train_losses)
plt.title("LSTM Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()


## 10. Make predictions on the test set


In [ ]:
model.eval()

with torch.no_grad():
    preds_scaled = model(X_test_t).cpu().numpy()

# Convert both predictions and actual values back to original demand scale
preds_actual = target_scaler.inverse_transform(preds_scaled)
y_test_actual = target_scaler.inverse_transform(y_test_seq)

preds_actual[:5], y_test_actual[:5]


## 11. Evaluate the LSTM on original demand values


In [ ]:
mae = mean_absolute_error(y_test_actual, preds_actual)
rmse = mean_squared_error(y_test_actual, preds_actual, squared=False)

# Avoid division by zero for MAPE
y_true_flat = y_test_actual.flatten()
y_pred_flat = preds_actual.flatten()
non_zero_mask = y_true_flat != 0
mape = (np.abs((y_true_flat[non_zero_mask] - y_pred_flat[non_zero_mask]) / y_true_flat[non_zero_mask]).mean()) * 100

results_df = pd.DataFrame([{
    "Model": "LSTM",
    "MAE": mae,
    "RMSE": rmse,
    "MAPE (%)": mape
}])

results_df


## 12. Plot actual vs predicted demand


In [ ]:
# The first prediction corresponds to the test set after seq_length steps
plot_index = test_df.index[seq_length:]

plt.figure(figsize=(15, 6))
plt.plot(plot_index, y_test_actual.flatten(), label="Actual")
plt.plot(plot_index, preds_actual.flatten(), label="LSTM Prediction")
plt.title("Actual vs Predicted Energy Demand using LSTM")
plt.xlabel("Time")
plt.ylabel(target_col)
plt.legend()
plt.show()


## 13. Inspect a few predictions


In [ ]:
comparison_df = pd.DataFrame({
    "Actual": y_test_actual.flatten(),
    "LSTM Prediction": preds_actual.flatten()
}, index=test_df.index[seq_length:])

comparison_df.head(20)


## Notes and suggestions

- This notebook avoids data leakage by splitting first, then scaling.
- The evaluation is done on the **original demand values**, not on scaled values.
- For fair comparison with ARIMA / SARIMA, use the same cleaned dataset and the same train-test period.
- You can improve this further by adding:
  - validation split
  - early stopping
  - more features such as weather variables
  - hyperparameter tuning for sequence length, hidden size, and epochs
